<a href="https://colab.research.google.com/github/SahasransuAcharjya/Text-to-Image-with-XLP-Pipeline/blob/main/text2image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Compatible libraries

In [1]:
!pip install "numpy<2.0"

In [2]:
# Install core AI libraries
!pip install -U diffusers transformers accelerate peft safetensors

# Install face extraction and processing tools
!pip install -q onnxruntime-gpu insightface opencv-python

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.


Backend Engine for the Model

In [2]:
import torch
import gc
import traceback
from PIL import Image
from diffusers import StableDiffusionXLPipeline
from diffusers.utils import load_image
import gradio as gr

print("Step 1: Building the Engine Class...")
class CustomImageEngine:
    def __init__(self):
        # 1. Load Base Pipeline directly to GPU. No CPU offloading!
        # This keeps everything inside the 16GB VRAM, avoiding System RAM crashes.
        self.pipe = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True
        ).to("cuda")

        self.ip_loaded = False
        print("Engine is built and ready in VRAM!")

    def _manage_ip_adapter(self, need_ip):
        """Smart function to load/unload Face Swapper instantly from local cache"""
        if need_ip and not self.ip_loaded:
            print("Loading IP-Adapter into VRAM...")
            self.pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
            self.ip_loaded = True
        elif not need_ip and self.ip_loaded:
            print("Unloading IP-Adapter to free VRAM...")
            self.pipe.unload_ip_adapter()
            self.ip_loaded = False

    def generate(self, prompt, negative_prompt=""):
        print(f"Generating Standard Image: {prompt}")
        self._manage_ip_adapter(need_ip=False)

        img = self.pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=30
        ).images[0]

        torch.cuda.empty_cache()
        gc.collect()
        return img

    def generate_with_face(self, prompt, face_img_path, negative_prompt=""):
        print(f"Generating Face Swap: {prompt}")
        self._manage_ip_adapter(need_ip=True)
        self.pipe.set_ip_adapter_scale(0.7)

        face_image = load_image(face_img_path).convert("RGB")
        img = self.pipe(
            prompt=prompt,
            ip_adapter_image=face_image,
            negative_prompt=negative_prompt,
            num_inference_steps=30
        ).images[0]

        torch.cuda.empty_cache()
        gc.collect()
        return img

print("Step 2: Initializing the models (this takes ~1 minute)...")
engine = CustomImageEngine()

print("Step 3: Launching the Web UI...")

# --- Gradio Interface Setup with Error Handling ---
def ui_generate(prompt, negative):
    if not prompt:
        raise gr.Error("Please enter a prompt first!")
    try:
        return engine.generate(prompt=prompt, negative_prompt=negative)
    except Exception as e:
        traceback.print_exc() # Prints exact error to Colab
        raise gr.Error(f"Generation failed: {str(e)}") # Shows error safely in UI

def ui_face_swap(prompt, face_img, negative):
    if not prompt:
        raise gr.Error("Please enter a prompt first!")
    if face_img is None:
        raise gr.Error("Please upload a face reference image!")
    try:
        return engine.generate_with_face(prompt=prompt, face_img_path=face_img, negative_prompt=negative)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Face swap failed: {str(e)}")

with gr.Blocks(theme=gr.themes.Soft()) as web_app:
    gr.Markdown("# AI Image Studio")
    gr.Markdown("Powered by SDXL & IP-Adapter")

    with gr.Tabs():
        # TAB 1: Standard Generation
        with gr.TabItem("Text to Image"):
            with gr.Row():
                with gr.Column():
                    prompt_in = gr.Textbox(label="Prompt", placeholder="A dog wearing sunglasses and lying next to a shore in rain...", lines=3)
                    neg_in = gr.Textbox(label="Negative Prompt", value="blurry, distorted, low quality, bad anatomy")
                    gen_btn = gr.Button("Generate", variant="primary")
                with gr.Column():
                    img_out = gr.Image(label="Output")

            gen_btn.click(fn=ui_generate, inputs=[prompt_in, neg_in], outputs=img_out)

        # TAB 2: Face Swapping
        with gr.TabItem("Face Swapping"):
            with gr.Row():
                with gr.Column():
                    face_prompt_in = gr.Textbox(label="Prompt", placeholder="A portrait of a person in a futuristic space suit...", lines=3)
                    face_upload = gr.Image(label="Upload Face Reference", type="filepath")
                    face_neg_in = gr.Textbox(label="Negative Prompt", value="blurry, distorted, low quality")
                    face_btn = gr.Button("Generate with Face", variant="primary")
                with gr.Column():
                    face_out = gr.Image(label="Output")

            face_btn.click(fn=ui_face_swap, inputs=[face_prompt_in, face_upload, face_neg_in], outputs=face_out)

web_app.launch(share=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Step 1: Building the Engine Class...
Step 2: Initializing the models (this takes ~1 minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Engine is built and ready in VRAM!
Step 3: Launching the Web UI...


/tmp/ipykernel_21293/3240725725.py:92: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as web_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20f2e666df394affac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
test_prompt = "A highly detailed, cinematic blueprint render of a bio-mimetic autonomous underwater vehicle exploring a deep sea trench, 8k resolution, photorealistic"
test_negative = "blurry, low quality, distorted, missing parts"

print("Generating image... (This first one might take ~45 seconds due to offloading)")

img = engine.generate(
    prompt=test_prompt,
    negative_prompt=test_negative
)

img

Generating image... (This first one might take ~45 seconds due to offloading)


NameError: name 'engine' is not defined